Import needed modules

In [37]:
import pandas as pd
import numpy as np
import os
import email

Function to get csv data as pandas `DataFrame`

In [4]:
EMAILS_DATASETS_PATH = os.path.join("datasets", "spam-emails")

def get_data(filename: str) -> pd.DataFrame:
    full_path = os.path.join(EMAILS_DATASETS_PATH, filename)
    return pd.read_csv(full_path)

emails_data = get_data("email_origin.csv")

Getting first entry to see overall structure of the email origin.

In [19]:
emails_data.iloc[0]

"Return-Path: <fork-admin@xent.com>\nDelivered-To: yyyy@localhost.netnoteinc.com\nReceived: from localhost (localhost [127.0.0.1])\n\tby phobos.labs.netnoteinc.com (Postfix) with ESMTP id 82A234416A\n\tfor <jm@localhost>; Mon, 26 Aug 2002 10:25:12 -0400 (EDT)\nReceived: from phobos [127.0.0.1]\n\tby localhost with IMAP (fetchmail-5.9.0)\n\tfor jm@localhost (single-drop); Mon, 26 Aug 2002 15:25:12 +0100 (IST)\nReceived: from xent.com ([64.161.22.236]) by dogma.slashnull.org\n    (8.11.6/8.11.6) with ESMTP id g7NKJVZ06924 for <jm@jmason.org>;\n    Fri, 23 Aug 2002 21:19:32 +0100\nReceived: from lair.xent.com (localhost [127.0.0.1]) by xent.com (Postfix)\n    with ESMTP id C9FFD2940C5; Fri, 23 Aug 2002 13:17:08 -0700 (PDT)\nDelivered-To: fork@spamassassin.taint.org\nReceived: from mail1.panix.com (mail1.panix.com [166.84.1.72]) by xent.com\n    (Postfix) with ESMTP id F1972294099 for <fork@xent.com>; Fri,\n    23 Aug 2002 13:16:24 -0700 (PDT)\nReceived: from 159-98.nyc.dsl.access.net (159

As we can see, there is 1897 spam adn 4151 non spam emails.

In [11]:
emails_data.info()
emails_data["label"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6048 entries, 0 to 6047
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   6048 non-null   int64 
 1   origin  6048 non-null   object
dtypes: int64(1), object(1)
memory usage: 94.6+ KB


label
0    4151
1    1897
Name: count, dtype: int64

Let's split this dataset into train and test sets.

In [20]:
from sklearn.model_selection import StratifiedShuffleSplit

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in sss.split(emails_data, emails_data["label"]):
    train, test = emails_data.iloc[train_idx], emails_data.iloc[test_idx]

Now, we need to dig into email structure and think how we can prepare and extract more data.

In [35]:
train["origin"].iloc[0]

'Return-Path: <ilug-admin@linux.ie>\nDelivered-To: zzzz@localhost.netnoteinc.com\nReceived: from localhost (localhost [127.0.0.1])\n\tby phobos.labs.netnoteinc.com (Postfix) with ESMTP id C2E7C47C74\n\tfor <zzzz@localhost>; Mon,  2 Sep 2002 07:42:03 -0400 (EDT)\nReceived: from phobos [127.0.0.1]\n\tby localhost with IMAP (fetchmail-5.9.0)\n\tfor zzzz@localhost (single-drop); Mon, 02 Sep 2002 12:42:03 +0100 (IST)\nReceived: from lugh.tuatha.org (root@lugh.tuatha.org [194.125.145.45]) by\n    dogma.slashnull.org (8.11.6/8.11.6) with ESMTP id g827VJZ23920 for\n    <zzzz-ilug@spamassassin.taint.org>; Mon, 2 Sep 2002 08:31:19 +0100\nReceived: from lugh (root@localhost [127.0.0.1]) by lugh.tuatha.org\n    (8.9.3/8.9.3) with ESMTP id IAA22561; Mon, 2 Sep 2002 08:30:44 +0100\nReceived: from moe.jinny.ie (homer.jinny.ie [193.120.171.3]) by\n    lugh.tuatha.org (8.9.3/8.9.3) with ESMTP id IAA22526 for <ilug@linux.ie>;\n    Mon, 2 Sep 2002 08:30:36 +0100\nX-Authentication-Warning: lugh.tuatha.org

In [53]:
message = email.message_from_string(train["origin"].iloc[0])